# Notebook 13 – Feature Engineering Pipeline

# Topic 1 – Feature Engineering Workflow



# Step 1 – Understand the Concept

A Feature Engineering Workflow is a sequence of steps used to prepare raw data for Machine Learning.

It helps convert raw data into meaningful features that improve model performance.

A typical workflow includes:

1. Load the dataset.
2. Understand the data.
3. Separate input features and target.
4. Split data into training and testing sets.
5. Create new features.
6. Transform numerical features.
7. Transform categorical features.
8. Select useful features.
9. Train and evaluate the model.

A structured workflow makes feature engineering reusable, organized, and reliable.



# Step 2 – Demonstrate the Concept

Real-world example: Loan Approval Prediction.

A bank collects customer information such as:

- Age
- Monthly Income
- Loan Amount
- Employment Type
- Credit Score
- Loan Approval Status

The raw data may not be directly suitable for Machine Learning.

We can create a new feature called Loan-to-Income Ratio.

Loan-to-Income Ratio = Loan Amount / Monthly Income

We can also scale numerical values and encode categorical values.

Business use:
The bank can use these prepared features to build a model that predicts whether a customer is likely to receive loan approval.



# Step 3 – Implement the Concept

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

data = {
    "Age": [25, 32, 45, 28, 50, 38, 29, 41, 35, 48,
            26, 39, 52, 31, 44, 27, 36, 47, 33, 42],
    "MonthlyIncome": [25000, 40000, 60000, 30000, 75000,
                      50000, 28000, 65000, 45000, 80000,
                      27000, 55000, 85000, 35000, 70000,
                      32000, 48000, 78000, 42000, 68000],
    "LoanAmount": [100000, 150000, 200000, 120000, 250000,
                   180000, 110000, 220000, 160000, 270000,
                   90000, 190000, 280000, 130000, 230000,
                   125000, 170000, 260000, 155000, 210000],
    "EmploymentType": [
        "Salaried", "Business", "Salaried", "Self-Employed",
        "Salaried", "Business", "Self-Employed", "Salaried",
        "Business", "Salaried", "Self-Employed", "Business",
        "Salaried", "Salaried", "Business", "Self-Employed",
        "Salaried", "Business", "Self-Employed", "Salaried"
    ],
    "CreditScore": [650, 700, 750, 620, 780, 710, 600, 760, 690, 800,
                    610, 720, 790, 640, 770, 630, 705, 785, 675, 740],
    "LoanApproved": [0, 1, 1, 0, 1, 1, 0, 1, 1, 1,
                     0, 1, 1, 0, 1, 0, 1, 1, 0, 1]
}

df = pd.DataFrame(data)

print("Dataset:")
display(df.head())

print("Dataset Shape:", df.shape)

Dataset:


,Age,MonthlyIncome,LoanAmount,EmploymentType,CreditScore,LoanApproved
0,25,25000,100000,Salaried,650,0
1,32,40000,150000,Business,700,1
2,45,60000,200000,Salaried,750,1
3,28,30000,120000,Self-Employed,620,0
4,50,75000,250000,Salaried,780,1


Dataset Shape: (20, 6)


# Output Explanation

- The dataset contains customer information.
- Each row represents one customer.
- LoanApproved is the target column.
- The remaining columns are input features.
- This dataset will be used throughout the notebook to build a reusable feature engineering pipeline.

# Topic 2 – Train/Test Separation


# Step 1 – Understand the Concept

Train/Test Separation means dividing a dataset into two parts:

1. Training Data – used to train the Machine Learning model.
2. Testing Data – used to evaluate the model on unseen data.

The training set teaches the model patterns from the data.

The testing set helps us understand how well the model performs on new data.

Important:
Split the data before fitting transformations that learn information from the dataset, such as imputers, scalers, encoders, or feature selectors.



# Step 2 – Demonstrate the Concept

Suppose a bank has 1,000 customer records.

We can divide them into:

- 80% Training Data
- 20% Testing Data

The model learns from the training data and is evaluated using the testing data.

Business example:
A bank wants to predict loan approval for new customers. The test data represents customers whose outcomes were not used to train the model.


# Step 3 – Implement the Concept

We will separate the input features (X) and target (y).

Then we will split the data into training and testing sets.

We use stratify=y to maintain a similar proportion of approved and rejected loans in both sets.

In [2]:
X = df.drop("LoanApproved", axis=1)
y = df["LoanApproved"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting Target Distribution:")
print(y_test.value_counts(normalize=True))

Training Features Shape: (16, 5)
Testing Features Shape: (4, 5)

Training Target Distribution:
LoanApproved
1    0.625
0    0.375
Name: proportion, dtype: float64

Testing Target Distribution:
LoanApproved
1    0.75
0    0.25
Name: proportion, dtype: float64


# Output Explanation

- X contains the customer input features.
- y contains the LoanApproved target.
- X_train and y_train are used for model training.
- X_test and y_test are used for model evaluation.
- stratify=y helps maintain the target class proportions in both sets.

The dataset is now separated before fitting any data-dependent preprocessing steps.

# Topic 3 – Pipeline


# Step 1 – Understand the Concept

A Pipeline is a Scikit-learn tool that combines multiple processing steps into one sequence.

Instead of performing every transformation separately, we can connect them together.

A Machine Learning pipeline can include:

1. Missing value handling.
2. Feature scaling.
3. Model training.

Benefits:

- Reusable workflow.
- Organized code.
- Consistent transformations.
- Helps prevent data leakage.
- Easier model training and testing.


# Step 2 – Demonstrate the Concept

Consider a customer loan prediction problem.

The model needs numerical data, but some customers may have missing income values.

We can create a pipeline that:

1. Fills missing values using the median.
2. Scales numerical values.
3. Trains a Logistic Regression model.

The pipeline performs these steps in the correct order whenever we call fit().



# Step 3 – Implement the Concept

We will create a simple numerical pipeline.

This is a small demonstration before building the complete reusable pipeline.

In [3]:
numerical_pipeline_demo = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

demo_numeric_columns = [
    "Age",
    "MonthlyIncome",
    "LoanAmount",
    "CreditScore"
]

numerical_pipeline_demo.fit(
    X_train[demo_numeric_columns],
    y_train
)

demo_predictions = numerical_pipeline_demo.predict(
    X_test[demo_numeric_columns]
)

print("Predictions:", demo_predictions)
print("Accuracy:", accuracy_score(y_test, demo_predictions))

Predictions: [1 1 1 1]
Accuracy: 0.75


# Output Explanation

- SimpleImputer fills missing numerical values using the median.
- StandardScaler standardizes numerical features.
- LogisticRegression trains the classification model.
- predict() uses the fitted pipeline to generate predictions.

All three steps are connected in one reusable object.

# Topic 4 – ColumnTransformer


# Step 1 – Understand the Concept

ColumnTransformer is a Scikit-learn tool used to apply different transformations to different columns.

A dataset can contain:

- Numerical columns: Age, MonthlyIncome, LoanAmount.
- Categorical columns: EmploymentType.

Numerical and categorical columns require different preprocessing methods.

ColumnTransformer allows us to process both types of columns in one object.



# Step 2 – Demonstrate the Concept

Loan prediction example:

Numerical features:
- Age
- MonthlyIncome
- LoanAmount
- CreditScore

Categorical feature:
- EmploymentType

We can apply:

- Median imputation and scaling to numerical columns.
- Most-frequent imputation and One-Hot Encoding to categorical columns.

ColumnTransformer combines these transformations.



# Step 3 – Implement the Concept

First, define the numerical and categorical column names.

Then create separate preprocessing pipelines and combine them using ColumnTransformer.

We will reuse these objects in the final feature engineering pipeline.

In [4]:
numeric_features = [
    "Age",
    "MonthlyIncome",
    "LoanAmount",
    "CreditScore"
]

categorical_features = [
    "EmploymentType"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("ColumnTransformer created successfully.")

ColumnTransformer created successfully.


# Output Explanation

- numeric_transformer handles numerical columns.
- categorical_transformer handles categorical columns.
- OneHotEncoder converts categories into numerical columns.
- handle_unknown="ignore" allows the pipeline to handle categories that were not seen during training.
- ColumnTransformer combines the outputs of both transformations.

The preprocessor is now ready to be reused.

# Topic 5 – Numerical Transformations



# Step 1 – Understand the Concept

Numerical Transformations prepare numerical data for Machine Learning.

Common techniques include:

1. Missing Value Imputation
2. Standardization
3. Normalization
4. Log Transformation
5. Square Root Transformation

In this notebook, we use:

- SimpleImputer to fill missing numerical values.
- StandardScaler to standardize numerical features.

StandardScaler transforms values using:

z = (x - mean) / standard deviation

It helps numerical features have a comparable scale.



# Step 2 – Demonstrate the Concept

Consider these customer details:

| Customer | Monthly Income | Loan Amount |
|---|---:|---:|
| A | 25,000 | 100,000 |
| B | 50,000 | 180,000 |
| C | 75,000 | 250,000 |

Income and loan amount have different ranges.

A feature with a much larger numerical range can influence some Machine Learning algorithms more strongly.

StandardScaler transforms numerical values to a common scale.

Business use:
This can help models such as Logistic Regression learn from customer features more consistently.



# Step 3 – Implement the Concept

We will apply the numerical transformation pipeline to the training data.

The imputer and scaler learn their values from training data only.

In [5]:
numeric_transformer.fit(X_train[numeric_features])

numeric_train = numeric_transformer.transform(
    X_train[numeric_features]
)

numeric_test = numeric_transformer.transform(
    X_test[numeric_features]
)

print("Transformed Training Shape:", numeric_train.shape)
print("Transformed Testing Shape:", numeric_test.shape)

print("\nFirst 5 Transformed Training Rows:")
print(numeric_train[:5])

Transformed Training Shape: (16, 4)
Transformed Testing Shape: (4, 4)

First 5 Transformed Training Rows:
[[-0.52311437 -0.46713322 -0.37434269  0.04899371]
 [-1.10435257 -0.88526645 -0.8083632  -1.04846549]
 [-0.17437146 -0.20579995 -0.20073449 -0.10778617]
 [-0.05812382 -0.04899999 -0.02712628  0.12738366]
 [ 0.17437146  0.05553332  0.14648192  0.2057736 ]]


# Output Explanation

- Missing numerical values are filled using the training median.
- The scaler learns the training mean and standard deviation.
- Training and testing data receive the same learned scaling.
- Both transformed datasets contain numerical values on a comparable scale.

We do not fit a separate scaler on the test data.

# Topic 6 – Categorical Transformations


# Step 1 – Understand the Concept

Categorical Transformations convert text-based categories into a format that Machine Learning algorithms can use.

Example:

EmploymentType:
- Salaried
- Business
- Self-Employed

A common technique is One-Hot Encoding.

One-Hot Encoding creates a separate binary column for each category.

For example:

EmploymentType_Salaried
EmploymentType_Business
EmploymentType_Self-Employed

Each column contains 1 or 0 depending on the customer's category.



# Step 2 – Demonstrate the Concept

Suppose a bank records customer employment types.

| Employment Type |
|---|
| Salaried |
| Business |
| Self-Employed |

After One-Hot Encoding:

| Salaried | Business | Self-Employed |
|---:|---:|---:|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |

Business use:
The bank's Machine Learning model can use employment categories as numerical inputs without assigning an artificial ranking to them.


# Step 3 – Implement the Concept

We will fit the categorical transformation pipeline using training data and apply it to both training and testing data.

In [6]:
categorical_transformer.fit(
    X_train[categorical_features]
)

categorical_train = categorical_transformer.transform(
    X_train[categorical_features]
)

categorical_test = categorical_transformer.transform(
    X_test[categorical_features]
)

print("Transformed Training Shape:", categorical_train.shape)
print("Transformed Testing Shape:", categorical_test.shape)

print("\nEncoded Training Data:")
print(categorical_train.toarray())

Transformed Training Shape: (16, 3)
Transformed Testing Shape: (4, 3)

Encoded Training Data:
[[1. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]


# Output Explanation

- SimpleImputer fills missing employment categories using the most frequent value.
- OneHotEncoder converts employment types into binary columns.
- The encoder learns categories from training data.
- The same encoder transforms testing data.

The output is now suitable for a Machine Learning model.

# Topic 7 – Feature Creation


# Step 1 – Understand the Concept

Feature Creation means generating new features from existing columns.

A new feature can provide additional information to a Machine Learning model.

Examples:

1. Loan-to-Income Ratio = Loan Amount / Monthly Income
2. Annual Income = Monthly Income × 12
3. Customer Age Group = Age-based category
4. Total Purchase = Quantity × Unit Price

Feature creation should be based on information that would actually be available when making a prediction.



# Step 2 – Demonstrate the Concept

Loan prediction example:

A customer has:

- Monthly Income = 30,000
- Loan Amount = 120,000

Loan-to-Income Ratio:

120,000 / 30,000 = 4

This means the requested loan amount is four times the customer's monthly income.

Business use:
A bank can use this ratio to understand the loan amount in relation to the customer's income.

Important:
If MonthlyIncome is zero or missing, we must handle division safely.



# Step 3 – Implement the Concept

We will create a reusable feature creation function.

The function will:

1. Copy the input DataFrame.
2. Create LoanToIncomeRatio.
3. Handle zero income safely.
4. Return the updated DataFrame.

We will later place this function inside the complete Scikit-learn pipeline.

In [7]:
from sklearn.preprocessing import FunctionTransformer

def create_features(data):
    data = data.copy()

    data["LoanToIncomeRatio"] = np.where(
        data["MonthlyIncome"] > 0,
        data["LoanAmount"] / data["MonthlyIncome"],
        np.nan
    )

    return data


feature_creator = FunctionTransformer(
    create_features,
    validate=False
)

X_train_created = feature_creator.transform(X_train)

print("Original Columns:")
print(X_train.columns.tolist())

print("\nNew Columns:")
print(X_train_created.columns.tolist())

display(X_train_created.head())

Original Columns:
['Age', 'MonthlyIncome', 'LoanAmount', 'EmploymentType', 'CreditScore']

New Columns:
['Age', 'MonthlyIncome', 'LoanAmount', 'EmploymentType', 'CreditScore', 'LoanToIncomeRatio']


,Age,MonthlyIncome,LoanAmount,EmploymentType,CreditScore,LoanToIncomeRatio
1,32,40000,150000,Business,700,3.750000
15,27,32000,125000,Self-Employed,630,3.906250
8,35,45000,160000,Business,690,3.555556
16,36,48000,170000,Salaried,705,3.541667
5,38,50000,180000,Business,710,3.600000


# Output Explanation

- The original dataset remains unchanged.
- LoanToIncomeRatio is added as a new feature.
- Zero income is handled by assigning a missing value instead of dividing by zero.
- The function can be reused on new customer records.

This feature creation function can now become a step in our complete pipeline.

# Topic 8 – Feature Selection


# Step 1 – Understand the Concept

Feature Selection means choosing the most useful input features for a Machine Learning model.

Not every feature provides useful information.

Some features may be:

- Relevant features: useful for prediction.
- Irrelevant features: provide little useful information.
- Redundant features: contain information similar to other features.

Feature selection can help:

1. Reduce unnecessary features.
2. Simplify the model.
3. Reduce training time.
4. Improve generalization in some cases.



# Step 2 – Demonstrate the Concept

Loan prediction example:

Suppose the dataset contains:

- Age
- Monthly Income
- Loan Amount
- Credit Score
- Employment Type
- Loan Approval Status

Some features may be more useful than others for predicting loan approval.

SelectPercentile is a Scikit-learn feature selection method.

It uses a statistical test to score features and keeps a selected percentage of the highest-scoring features.

We will use the ANOVA F-test (f_classif) for this classification example.

Note:
Feature selection must be fitted using training data only.



# Step 3 – Implement the Concept

We will create a feature selection step using SelectPercentile.

The percentile value is set to 80, meaning the selector keeps approximately 80% of the transformed features.

The selector will be fitted later as part of the complete pipeline.

In [8]:
feature_selector = SelectPercentile(
    score_func=f_classif,
    percentile=80
)

print("Feature Selector:")
print(feature_selector)

print("\nSelected Percentage:", feature_selector.percentile)

Feature Selector:
SelectPercentile(percentile=80)

Selected Percentage: 80


# Output Explanation

- SelectPercentile selects features based on their statistical scores.
- f_classif is suitable for classification problems with numerical input features.
- percentile=80 keeps approximately 80% of the available features.
- The actual selected features depend on the training data.

The selector has been configured but has not learned from the data yet.
It will learn when the complete pipeline is fitted.

# Topic 9 – Transformation Pipeline



# Step 1 – Understand the Concept

A Transformation Pipeline combines multiple feature engineering steps into one reusable workflow.

Our complete pipeline will contain:

1. Feature Creation
2. ColumnTransformer
3. Numerical Transformations
4. Categorical Transformations
5. Feature Selection
6. Machine Learning Model

The pipeline applies these steps in the correct order.

Once created, the same pipeline can be used for training, testing, and making predictions on new customer records.



# Step 2 – Demonstrate the Concept

Loan prediction workflow:

Raw Customer Data
        |
        v
Feature Creation
        |
        v
ColumnTransformer
        |
        +----------------------+
        |                      |
        v                      v
Numerical Processing     Categorical Processing
        |                      |
        +----------+-----------+
                   |
                   v
            Feature Selection
                   |
                   v
             Logistic Regression
                   |
                   v
             Loan Prediction

This complete sequence is reusable.




# Step 3 – Implement the Concept

We will now build the complete pipeline.

First, add the newly created LoanToIncomeRatio feature to the numerical column list.

Then combine feature creation, preprocessing, feature selection, and Logistic Regression into one Pipeline.

The feature creator runs before ColumnTransformer, so the new feature is available for numerical processing.

In [9]:
numeric_features = [
    "Age",
    "MonthlyIncome",
    "LoanAmount",
    "CreditScore",
    "LoanToIncomeRatio"
]

categorical_features = [
    "EmploymentType"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

complete_pipeline = Pipeline(steps=[
    ("feature_creation", feature_creator),
    ("preprocessing", preprocessor),
    ("feature_selection", SelectPercentile(
        score_func=f_classif,
        percentile=80
    )),
    ("model", LogisticRegression(max_iter=1000))
])

print("Complete Feature Engineering Pipeline:")
print(complete_pipeline)

Complete Feature Engineering Pipeline:
Pipeline(steps=[('feature_creation',
                 FunctionTransformer(func=<function create_features at 0x00000262FA2BCF40>)),
                ('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'MonthlyIncome',
                                                   'LoanAmount', 'CreditScore',
                                                   'LoanToIncomeRatio']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                              

# Output Explanation

The complete pipeline contains four main stages:

1. feature_creation:
   Creates LoanToIncomeRatio.

2. preprocessing:
   Imputes and scales numerical features.
   Imputes and encodes categorical features.

3. feature_selection:
   Selects useful transformed features using the ANOVA F-test.

4. model:
   Trains a Logistic Regression classifier.

All these steps are connected in one reusable pipeline.

# Step 3 – Train the Complete Pipeline

We will fit the complete pipeline using only the training data.

During fit():

- Feature creation is applied.
- Imputation values are learned.
- Scaling values are learned.
- Categories are learned by the encoder.
- Feature selection is fitted.
- Logistic Regression is trained.

The test data will not be used to learn these values.

In [10]:
complete_pipeline.fit(X_train, y_train)

print("Pipeline training completed successfully.")

Pipeline training completed successfully.


# Output Explanation

The pipeline has successfully learned its transformations and trained the model using training data.

The fitted pipeline now contains:

- Learned imputation values.
- Learned scaling parameters.
- Learned categorical encoding.
- Selected features.
- A trained Logistic Regression model.

We can now use this same fitted pipeline to predict loan approval for the test data.

# Step 3 – Make Predictions and Evaluate

We will pass the original test data into the fitted pipeline.

The pipeline automatically performs feature creation and preprocessing before generating predictions.

We will evaluate the model using accuracy and a classification report.

In [11]:
y_pred = complete_pipeline.predict(X_test)

print("Predictions:")
print(y_pred)

print("\nActual Values:")
print(y_test.to_numpy())

print("\nAccuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

Predictions:
[1 1 1 1]

Actual Values:
[1 0 1 1]

Accuracy:
0.75

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.75      1.00      0.86         3

    accuracy                           0.75         4
   macro avg       0.38      0.50      0.43         4
weighted avg       0.56      0.75      0.64         4



# Output Explanation

- predict() applies all fitted transformations to X_test.
- The model generates loan approval predictions.
- Accuracy shows the proportion of correct predictions.
- The classification report displays precision, recall, and F1-score.

Because this is a very small sample dataset, the evaluation score is only for learning the pipeline workflow. It should not be treated as evidence of real-world model performance.

# Topic 10 – Preventing Data Leakage



# Step 1 – Understand the Concept

Data Leakage happens when information from outside the training process becomes available to the model during training.

This can make model performance look better than it really is.

Common examples:

1. Scaling the complete dataset before train/test splitting.
2. Filling missing values using the complete dataset.
3. Selecting features using both training and testing targets.
4. Using future information to predict a past event.

Ways to prevent leakage:

- Split data before fitting transformations.
- Fit preprocessing steps only on training data.
- Use Pipeline and ColumnTransformer.
- Apply the fitted transformations to test data.
- Use only information available at prediction time.



# Step 2 – Demonstrate the Concept

Consider a bank predicting loan approval.

Incorrect approach:

1. Calculate the average income using the complete dataset.
2. Scale all customer records.
3. Split the data into training and testing sets.
4. Train the model.

Problem:
Information from the test data has influenced the preprocessing.

Correct approach:

1. Split the data into training and testing sets.
2. Fit preprocessing using training data only.
3. Train the model using training data.
4. Apply the fitted transformations to test data.

Our complete_pipeline follows the correct approach because the preprocessing and feature selection steps are fitted during pipeline.fit(X_train, y_train).




# Step 3 – Implement the Concept

We will verify that our pipeline is fitted using training data and then used to transform the test data.

We will also compare the scaler's learned mean with the mean of the training data.

In [15]:



# Get the fitted numerical scaler from the pipeline
fitted_preprocessor = complete_pipeline.named_steps["preprocessing"]

fitted_numeric_pipeline = (
    fitted_preprocessor.named_transformers_["num"]
)

fitted_scaler = fitted_numeric_pipeline.named_steps["scaler"]

# Create the new feature separately for checking
X_train_created = feature_creator.transform(X_train)

# Compare training means with scaler's learned means
training_means = X_train_created[numeric_features].mean()

print("Training Data Means:")
print(training_means)

print("\nScaler Learned Means:")
print(pd.Series(
    fitted_scaler.mean_,
    index=numeric_features
))

# Transform test data using the fitted pipeline
test_predictions = complete_pipeline.predict(X_test)

print("\nTest Predictions:")
print(test_predictions)


Training Data Means:
Age                      36.500000
MonthlyIncome         48937.500000
LoanAmount           171562.500000
CreditScore             696.875000
LoanToIncomeRatio         3.585253
dtype: float64

Scaler Learned Means:
Age                      36.500000
MonthlyIncome         48937.500000
LoanAmount           171562.500000
CreditScore             696.875000
LoanToIncomeRatio         3.585253
dtype: float64

Test Predictions:
[1 1 1 1]


# Output Explanation

- The scaler's learned mean values come from the training data after feature creation and imputation.
- The test data is transformed using the already fitted preprocessing steps.
- The pipeline does not calculate new scaling values from the test data.
- Feature selection is also fitted using training data only.
- The test target is not used during pipeline fitting.

This helps prevent preprocessing-related data leakage.

# Complete Feature Engineering Workflow

Our reusable pipeline follows this sequence:

1. Load the customer dataset.
2. Separate input features and target.
3. Split the data into training and testing sets.
4. Create new features.
5. Apply numerical transformations.
6. Apply categorical transformations.
7. Combine transformations using ColumnTransformer.
8. Select useful features.
9. Train the Logistic Regression model.
10. Evaluate predictions using test data.

The entire process is managed by one Scikit-learn Pipeline.

In [13]:
print("FINAL PIPELINE STEPS\n")

for step_name, step_object in complete_pipeline.steps:
    print(f"{step_name} -> {type(step_object).__name__}")

print("\nTraining Accuracy:",
      complete_pipeline.score(X_train, y_train))

print("Testing Accuracy:",
      complete_pipeline.score(X_test, y_test))

FINAL PIPELINE STEPS

feature_creation -> FunctionTransformer
preprocessing -> ColumnTransformer
feature_selection -> SelectPercentile
model -> LogisticRegression

Training Accuracy: 1.0
Testing Accuracy: 0.75


# Final Output Explanation

The final output displays the steps included in our reusable pipeline.

Training Accuracy:
Shows how well the model predicts the training data.

Testing Accuracy:
Shows how well the model predicts unseen testing data.

A large difference between training and testing performance may indicate overfitting or other modelling problems.

The pipeline can now be reused for new customer data without manually repeating every feature engineering step.

# Feature Engineering Pipeline – Checklist

- [x] Feature Engineering Workflow
- [x] Train/Test Separation
- [x] Pipeline
- [x] ColumnTransformer
- [x] Numerical Transformations
- [x] Categorical Transformations
- [x] Feature Creation
- [x] Feature Selection
- [x] Transformation Pipeline
- [x] Preventing Data Leakage

## Key Learnings

1. A Pipeline combines multiple Machine Learning steps.
2. ColumnTransformer applies different transformations to different columns.
3. Feature creation generates meaningful features from existing data.
4. Feature selection keeps useful features.
5. Train/test separation must happen before fitting data-dependent transformations.
6. A fitted pipeline can be reused for new customer records.

## Conclusion

We successfully built a reusable Feature Engineering Pipeline using Pandas and Scikit-learn.

The pipeline performs feature creation, numerical and categorical preprocessing, feature selection, and Logistic Regression model training in one workflow.

It also helps prevent data leakage by learning preprocessing parameters from training data only.